In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import joblib
import os
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from sklearn.ensemble import GradientBoostingRegressor
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


def engineer_features_a(df):
    df = df.copy()
    df["stress_x_anxiety"] = df["stress_level"] * df["anxiety_score"]
    df["stress_x_depression"] = df["stress_level"] * df["depression_score"]
    df["stress_x_exam"] = df["stress_level"] * df["exam_pressure"]
    df["anxiety_x_depression"] = df["anxiety_score"] * df["depression_score"]
    df["study_sleep_ratio"] = df["study_hours_per_day"] / (df["sleep_hours"] + 1)
    df["pressure_support_gap"] = (
        (df["exam_pressure"] + df["financial_stress"] + df["family_expectation"]) / 3
    ) - df["social_support"]
    df["stress_level_sq"] = df["stress_level"] ** 2
    df["anxiety_score_sq"] = df["anxiety_score"] ** 2
    df["sleep_deprived"] = (df["sleep_hours"] < 5).astype(int)
    df["high_stress"] = (df["stress_level"] > 7).astype(int)
    return df


mlflow.set_experiment("Academic Shield - Burnout & GPA Prediction")

with mlflow.start_run(run_name="Model_A_XGBoost_Classifier"):

    print("Model A - XGBoost Classifier")

    mlflow.set_tags(
        {
            "model_type": "XGBoost Classifier",
            "task": "Burnout Classification",
            "dataset": "academic_stress_level.csv (1M rows)",
            "target": "burnout_class {0 = Healthy, 1 = Mildly Burnout, 2 = Burnout}",
            "thresholds": "(score < 4) = Healthy, (4-7) = Mild, (>= 7) = Burnout",
            "features": "10 base features → 20 features after feature engineering",
            "cleaning": "IQR on feature columns only (preserve minority class)",
            "balancing": "SMOTE on training set only",
            "primary_metric": "Macro F1",
            "tuning": "Optuna TPE (150 trials)",
            "team": "Group 3 - Andrew/Raynald/Adrian",
        }
    )

    df_a = pd.read_csv("academic_stress_level.csv")

    cols_a = [
        "study_hours_per_day",
        "sleep_hours",
        "exam_pressure",
        "stress_level",
        "financial_stress",
        "social_support",
        "anxiety_score",
        "depression_score",
        "family_expectation",
        "physical_activity",
        "burnout_score",
    ]
    df_a = df_a[cols_a]

    def bin_burnout(score):
        if score < 4:
            return 0  # Healthy
        elif score < 7:
            return 1  # Mildly Burnout
        else:
            return 2  # Burnout

    df_a["burnout_class"] = df_a["burnout_score"].apply(bin_burnout)
    df_a = df_a.drop(columns=["burnout_score"])

    print(f"Class distribution:\n{df_a['burnout_class'].value_counts().to_string()}")

    feature_cols_a = [c for c in df_a.columns if c != "burnout_class"]
    for col in feature_cols_a:
        q1 = df_a[col].quantile(0.25)
        q3 = df_a[col].quantile(0.75)
        iqr = q3 - q1
        df_a = df_a[(df_a[col] >= q1 - 1.5 * iqr) & (df_a[col] <= q3 + 1.5 * iqr)]
    print(f"After cleaning: {len(df_a)} rows")
    print(
        f"Class distribution after cleaning:\n{df_a['burnout_class'].value_counts().to_string()}"
    )

    X_a = engineer_features_a(df_a.drop(columns=["burnout_class"]))
    y_a = df_a["burnout_class"]
    print(f"Features: {X_a.shape[1]}")

    X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
        X_a, y_a, test_size=0.2, random_state=42, stratify=y_a
    )
    smote = SMOTE(random_state=42)
    X_train_a_sm, y_train_a_sm = smote.fit_resample(X_train_a, y_train_a)
    print(f"Train before SMOTE: {X_train_a.shape[0]} rows")
    print(f"Train after SMOTE : {X_train_a_sm.shape[0]} rows")
    print(f"Test              : {X_test_a.shape[0]} rows")

    best_params_a = {
        "n_estimators": 2019,
        "max_depth": 12,
        "learning_rate": 0.0864,
        "min_child_weight": 1,
        "subsample": 0.8746,
        "colsample_bytree": 0.7718,
        "reg_alpha": 0.0255,
        "reg_lambda": 0.1374,
        "eval_metric": "mlogloss",
        "random_state": 42,
    }
    mlflow.log_params(
        {
            **best_params_a,
            "tuning_trials": 150,
            "train_test_split": "80/20 stratified",
            "smote": "applied to training set only",
            "iqr_cleaning": "features only",
            "n_features_base": 10,
            "n_features_total": 20,
            "original_size": 1000000,
        }
    )

    model_a = XGBClassifier(
        n_estimators=best_params_a["n_estimators"],
        max_depth=best_params_a["max_depth"],
        learning_rate=best_params_a["learning_rate"],
        min_child_weight=best_params_a["min_child_weight"],
        subsample=best_params_a["subsample"],
        colsample_bytree=best_params_a["colsample_bytree"],
        reg_alpha=best_params_a["reg_alpha"],
        reg_lambda=best_params_a["reg_lambda"],
        eval_metric="mlogloss",
        random_state=42,
    )
    model_a.fit(X_train_a_sm, y_train_a_sm)

    y_pred_a = model_a.predict(X_test_a)

    test_acc_a = accuracy_score(y_test_a, y_pred_a)
    test_f1_a = f1_score(y_test_a, y_pred_a, average="macro")
    test_prec_a = precision_score(y_test_a, y_pred_a, average="macro", zero_division=0)
    test_rec_a = recall_score(y_test_a, y_pred_a, average="macro", zero_division=0)

    report_a = classification_report(
        y_test_a,
        y_pred_a,
        target_names=["Healthy", "Mildly Burnout", "Burnout"],
        output_dict=True,
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1_a = cross_val_score(model_a, X_a, y_a, cv=skf, scoring="f1_macro")
    cv_acc_a = cross_val_score(model_a, X_a, y_a, cv=skf, scoring="accuracy")

    dummy_acc_a = y_a.value_counts(normalize=True).max()
    dummy_f1_a = 1 / 3

    mlflow.log_metrics(
        {
            "test_accuracy": round(test_acc_a, 4),
            "test_macro_f1": round(test_f1_a, 4),
            "test_precision_macro": round(test_prec_a, 4),
            "test_recall_macro": round(test_rec_a, 4),
            "f1_healthy": round(report_a["Healthy"]["f1-score"], 4),
            "f1_mild": round(report_a["Mildly Burnout"]["f1-score"], 4),
            "f1_burnout": round(report_a["Burnout"]["f1-score"], 4),
            "cv_accuracy_mean": round(cv_acc_a.mean(), 4),
            "cv_accuracy_std": round(cv_acc_a.std(), 4),
            "cv_macro_f1_mean": round(cv_f1_a.mean(), 4),
            "cv_macro_f1_std": round(cv_f1_a.std(), 4),
            "dummy_accuracy": round(float(dummy_acc_a), 4),
            "dummy_macro_f1": round(dummy_f1_a, 4),
        }
    )

    mlflow.xgboost.log_model(model_a, name="model_a")

    import cloudpickle

    with open("pipeline_a_temp.pkl", "wb") as f:
        cloudpickle.dump(engineer_features_a, f)
    mlflow.log_artifact("pipeline_a_temp.pkl", artifact_path="pipeline")
    os.remove("pipeline_a_temp.pkl")

    class_map = {0: "Healthy", 1: "Mildly Burnout", 2: "Burnout"}
    joblib.dump(class_map, "class_map_temp.pkl")
    mlflow.log_artifact("class_map_temp.pkl", artifact_path="pipeline")
    os.remove("class_map_temp.pkl")

    print(f"\n  Test Accuracy  : {test_acc_a:.4f}")
    print(f"  Test Macro F1  : {test_f1_a:.4f}")
    print(f"  CV Macro F1    : {cv_f1_a.mean():.4f} ± {cv_f1_a.std():.4f}")
    print(f"  Dummy baseline : {dummy_acc_a:.4f} acc / ~{dummy_f1_a:.2f} F1")
    print("\nModel A logged successfully.")

with mlflow.start_run(run_name="Model_B_GradientBoosting_Regressor"):

    print("Model B - GradientBoosting Regressor")

    mlflow.set_tags(
        {
            "model_type": "GradientBoostingRegressor",
            "task": "GPA Regression",
            "dataset": "student_lifestyle_dataset.csv (~2000 rows)",
            "target": "GPA (0.0 - 4.0)",
            "cleaning": "Z-score threshold = 3 (conservative for small dataset)",
            "encoding": "stress_level: Low = 0, Moderate = 1, High = 2",
            "primary_metric": "R2",
            "team": "Group 3 - Andrew/Raynald/Adrian",
        }
    )

    df_b = pd.read_csv("student_lifestyle_dataset.csv")
    df_b = df_b.drop(columns=["Student_ID"])
    df_b = df_b.rename(
        columns={
            "Study_Hours_Per_Day": "study_hours",
            "Extracurricular_Hours_Per_Day": "eca_hours",
            "Sleep_Hours_Per_Day": "sleep_hours",
            "Social_Hours_Per_Day": "social_hours",
            "Physical_Activity_Hours_Per_Day": "physical_hours",
            "Stress_Level": "stress_level",
            "GPA": "gpa",
        }
    )

    print(f"Rows: {len(df_b)}")

    df_b = df_b[(np.abs(stats.zscore(df_b.select_dtypes("number"))) < 3).all(axis=1)]
    print(f"After cleaning: {len(df_b)} rows")

    stress_mapping = {"Low": 0, "Moderate": 1, "High": 2}
    df_b["stress_level"] = df_b["stress_level"].map(stress_mapping)

    X_b = df_b.drop("gpa", axis=1)
    y_b = df_b["gpa"]
    X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
        X_b, y_b, test_size=0.3, random_state=42
    )

    params_b = {
        "n_estimators": 1000,
        "max_depth": 2,
        "learning_rate": 0.01,
        "subsample": 0.9,
        "max_features": 0.7,
        "min_samples_leaf": 3,
        "n_iter_no_change": 50,
        "validation_fraction": 0.1,
        "tol": 1e-4,
        "random_state": 42,
    }

    mlflow.log_params(
        {
            **params_b,
            "train_test_split": "70/30",
            "cleaning_method": "Z-score threshold = 3",
            "encoding": "manual mapping (Low = 0/Moderate = 1/High = 2)",
            "n_features": 6,
            "dataset_size": len(df_b),
            "feature_engineering": "None (simple model)",
        }
    )

    model_b = GradientBoostingRegressor(**params_b)
    model_b.fit(X_train_b, y_train_b)
    y_pred_b = model_b.predict(X_test_b)

    mse_b = mean_squared_error(y_test_b, y_pred_b)
    rmse_b = np.sqrt(mse_b)
    mae_b = mean_absolute_error(y_test_b, y_pred_b)
    r2_b = r2_score(y_test_b, y_pred_b)

    actual_range_b = y_test_b.max() - y_test_b.min()
    nrmse_b = rmse_b / actual_range_b

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2_b = cross_val_score(model_b, X_b, y_b, cv=kf, scoring="r2")
    cv_mae_b = cross_val_score(
        model_b, X_b, y_b, cv=kf, scoring="neg_mean_absolute_error"
    )
    cv_rmse_b = cross_val_score(
        model_b, X_b, y_b, cv=kf, scoring="neg_root_mean_squared_error"
    )

    mlflow.log_metrics(
        {
            "test_mse": round(mse_b, 4),
            "test_rmse": round(rmse_b, 4),
            "test_mae": round(mae_b, 4),
            "test_r2": round(r2_b, 4),
            "nrmse_pct": round(nrmse_b * 100, 2),
            "actual_gpa_range": round(float(actual_range_b), 4),
            "cv_r2_mean": round(cv_r2_b.mean(), 4),
            "cv_r2_std": round(cv_r2_b.std(), 4),
            "cv_mae_mean": round((-cv_mae_b).mean(), 4),
            "cv_mae_std": round((-cv_mae_b).std(), 4),
            "cv_rmse_mean": round((-cv_rmse_b).mean(), 4),
            "cv_rmse_std": round((-cv_rmse_b).std(), 4),
        }
    )

    mlflow.sklearn.log_model(model_b, name="model_b")

    joblib.dump(stress_mapping, "encoder_temp.pkl")
    mlflow.log_artifact("encoder_temp.pkl", artifact_path="encoder")
    os.remove("encoder_temp.pkl")

    print(f"\n  Test R²    : {r2_b:.4f}")
    print(f"  Test RMSE  : {rmse_b:.4f}")
    print(f"  NRMSE      : {nrmse_b*100:.2f}%")
    print(f"  CV R²      : {cv_r2_b.mean():.4f} ± {cv_r2_b.std():.4f}")
    print("\nModel B logged successfully.")

print("  Academic Shield - MLflow Pipeline Summary")
print("  Experiment : Academic Shield - Burnout & GPA Prediction")
print(f"  Run 1 (A)  : XGBoost Classifier")
print(f"               Test Macro F1 : {test_f1_a:.4f}")
print(f"               CV Macro F1   : {cv_f1_a.mean():.4f} ± {cv_f1_a.std():.4f}")
print(f"  Run 2 (B)  : GradientBoosting Regressor")
print(f"               Test R²       : {r2_b:.4f}")
print(f"               NRMSE         : {nrmse_b*100:.2f}%")

2026/05/28 20:16:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model A logged successfully.


2026/05/28 20:16:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 20:16:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model B logged successfully.
Academic Shield — MLflow Experiment Summary
  Experiment : Academic Shield - Burnout & GPA Prediction
  Run 1      : Model A — XGBoost Classifier
               CV Accuracy : 0.9233 ± 0.0005
               CV Macro F1 : 0.5373 ± 0.0049
               Dummy F1    : ~0.31 (baseline)
  Run 2      : Model B — GradientBoosting Regressor
               R²          : 0.5333
               NRMSE       : 11.94%
To view results: mlflow ui
To zip artifacts: !zip -r mlruns.zip mlruns/
